# Mirror Test — cross-family extension (Gemma-2, Mistral, Llama)

**Build `2026-07-23-f3106874`.** One notebook, runnable end-to-end on **Kaggle GPU T4 ×2**,
that extends the single-family (Qwen2.5) study to more open training recipes **without
touching any published number**. Every new measurement reuses the repository's own
functions, config, seeds, and templates, so new cells are directly comparable to the
released ones.

## Why this exists
The paper's central claims — the **implicit/explicit dissociation** and the **scale
trend** — are currently supported on one family (Qwen2.5). A reviewer's first move is
"that's a Qwen case study, not a general result." This notebook answers that:

* **Tier 1 — generalize the DISSOCIATION** (≈ pure inference). Promote three existing
  *foil* families to *judges* at their current size — `gemma-2-9b-it`,
  `mistral-7b-instruct-v0.3`, `llama-3.2-3b-instruct`. Their generations already exist,
  so the only new generation is each one's **placebo second sample** (`s2`). We then run
  them as PPP + IPP judges, compute per-token NLL for the perplexity rule, and add
  placebo + stylometry. Target: *near-perfect implicit self-info, chance-level explicit
  choice, κ≈0* replicates on **4 recipes** (Qwen + these three).
* **Tier 2 — generalize the SCALE CURVE** (moderate new generation). A full Gemma-2 size
  sweep — `gemma-2-2b-it` (generate), `gemma-2-9b-it` (reuse from Tier 1),
  `gemma-2-27b-it` (generate; sharded across both T4s) — judged and plotted **beside**
  Qwen's curve.

We deliberately **do not** re-run the paraphrase attack or the base-vs-instruct
ablation (already established on Qwen); compute goes to judging, perplexity, and
stylometry for the new judges.

## Common foil pool (comparability)
Pool = `{llama-3.2-3b-instruct, gemma-2-9b-it, mistral-7b-instruct-v0.3, human}`. Each
judge's foils = the pool families that are **not its own family**, plus `human`. All foil
text already exists — **foils never need new generation.**

| Judge | Foils |
|---|---|
| gemma-2-{2b,9b,27b}-it | llama-3.2-3b-instruct, mistral-7b-instruct-v0.3, human |
| mistral-7b-instruct-v0.3 | llama-3.2-3b-instruct, gemma-2-9b-it, human |
| llama-3.2-3b-instruct | gemma-2-9b-it, mistral-7b-instruct-v0.3, human |
| qwen2.5-* (unchanged, reused) | llama, gemma-2-9b, mistral, human |

## Kaggle setup — just upload this one file and run
No dataset to attach and nothing to unzip: the notebook **clones the repo itself** (code +
frozen data + published results) and writes everything to `/kaggle/working/`.

1. **Push the repo first** (public) to the URL in the config cell's `REPO_GIT_URL` — the
   notebook clones it. (Private repo? add a `GITHUB_TOKEN` secret.) Make sure the push
   includes the `src/utils.py` `MIRROR_MAX_MEMORY` sharding path — the 27B judge needs it.
2. New Kaggle notebook → **Accelerator = GPU T4 ×2**, **Internet = ON**.
3. **Add secret `HF_TOKEN`** (Add-ons → Secrets), and with that same Hugging Face account
   **accept the licenses** for `google/gemma-2-2b-it`, `google/gemma-2-9b-it`,
   `google/gemma-2-27b-it`, `meta-llama/Llama-3.2-3B-Instruct` — otherwise the gated
   downloads 403.
4. **Upload `kaggle_extend_families.ipynb`** and **Run All**. That's it.

The notebook is **checkpointed and resumable**: rerun in a fresh session and it skips every
unit already complete. Tier 1 + Tier 2 may span more than one 12 h session; 27B is scheduled
**last** so the rest ships even if it slips. (You can still attach the repo as a Dataset
instead of cloning — it is auto-detected — but nothing requires it.)

## Outputs (to `/kaggle/working/`)
* `extended_table1.csv` — per (judge, foil): n, PPP acc + Wilson CI, AUROC + CI, stylometry
  acc, perplexity-rule acc, κ, Holm-adjusted significance (recomputed over **all** cells).
* `dissociation_summary.csv` — per judge: implicit (PPL-rule) vs explicit (PPP) accuracy
  (both pooled over the same LLM-foil universe), the **implicit−explicit accuracy gap**
  (the primary, non-saturating dissociation evidence), κ, and the margin–margin Spearman ρ
  (a companion to κ that does not collapse under marginal saturation).
* `scale_curves.png` — Qwen2.5 vs Gemma-2 for PPP accuracy, PPL-rule accuracy, and
  human-vs-machine AUROC, vs log judge size.
* `extended_outputs/raw/…` — raw per-cell judgments (JSONL + parquet if available).
* `run_report.txt` — timings, GPU-hours, reused vs generated, skipped/failed, resolved
  model revisions, Holm family size, per-cell n + power note, library versions.

> **Scope honesty (kept in the outputs).** Two new families + a one-family scale sweep is
> a targeted generalization, not a universal law: claims are scoped to *open
> instruction-tuned models*, the paraphrase mechanism result stays Qwen-only, and every
> accuracy ships with a CI so the reader judges consistency, not just significance.


In [ ]:
# ============================== RUN SETTINGS ================================
# The ONE place to change paths / models / budget / pool. Everything below and
# in later cells reads these globals. Nothing experiment-relevant is hidden in
# the other cells. Printed at the end so a reviewer can audit it at a glance.
#
# Design rule (matches the repo): seeds, decoding params, prompt templates and
# statistical settings are NOT redefined here — they are read from the frozen
# configs/models.yaml in the attached dataset, so the new cells are identical
# to the published ones. This cell only adds the NEW judges + the foil pool +
# the Kaggle plumbing the repo cannot know about.
# ===========================================================================

import os

# ---- Where the repo + data come from ---------------------------------------
# Self-contained, upload-and-run (like kaggle_run_all): the bootstrap cell
# CLONES the pushed repo — code + frozen data + published results — so there is
# NO Kaggle Dataset to attach. The notebook already needs Internet for the model
# weights, so cloning adds no new requirement. If you DO attach the repo as a
# Dataset it is auto-detected and used without cloning; INPUT_ROOT_OVERRIDE
# forces a specific local path (used by LOCAL_TEST).
REPO_GIT_URL        = "https://github.com/shehryars715/MirrorTest.git"
REPO_GIT_REF        = "main"        # branch / tag / commit to clone
INPUT_ROOT_OVERRIDE = None          # force an attached/local repo dir instead of cloning

WORKING_ROOT        = "/kaggle/working"        # writable; MIRROR_ROOT points here
CHECKPOINT_DIR      = "/kaggle/working/checkpoints"
OUTPUT_DIR          = "/kaggle/working/extended_outputs"
# HF weight cache: deliberately NOT under /kaggle/working — that mount has a
# ~20 GB persisted cap and gemma-2-27b-it alone is ~54 GB. /kaggle/temp is the
# large ephemeral scratch. Bootstrap falls back to the default ~/.cache if this
# is not writable.
HF_CACHE_DIR        = "/kaggle/temp/hf_cache"

# ---- Session control -------------------------------------------------------
BUDGET_HOURS = 11.0     # stop launching new work after this; resume next session
DRY_RUN      = False    # True = print the plan and skip all GPU work
ENABLE_27B   = True     # gemma-2-27b-it (sharded 2xT4). Set False to cap Gemma-2
                        # at 2B/9B and guarantee single-session completion (§18
                        # of the protocol sanctions a truncated scale axis).
RUN_STYLOMETRY = True   # CPU char n-gram baseline for the new cells
RUN_IPP        = True   # Yes/No protocol for the new judges

# LOCAL_TEST lets the CPU-only cells (bootstrap file-checks, stats, plotting)
# be exercised off-Kaggle against a normal checkout. When True, INPUT_ROOT and
# WORKING_ROOT are taken from the two env vars below and all GPU tasks are
# skipped. On Kaggle leave this False.
LOCAL_TEST = bool(os.environ.get("MIRROR_LOCAL_TEST"))
if LOCAL_TEST:
    INPUT_ROOT_OVERRIDE = os.environ["MIRROR_LOCAL_INPUT"]      # repo parent dir
    WORKING_ROOT        = os.environ["MIRROR_LOCAL_WORKING"]
    CHECKPOINT_DIR      = WORKING_ROOT + "/checkpoints"
    OUTPUT_DIR          = WORKING_ROOT + "/extended_outputs"
    HF_CACHE_DIR        = WORKING_ROOT + "/hf_cache"
    DRY_RUN = True

# ---- Statistics ------------------------------------------------------------
# The published cell_stats() uses 500 resamples for the Table-1 AUROC CI and
# the config's 1000 for kappa. We recompute EVERY cell (old + new) here with a
# single uniform N so all rows in the extended outputs are mutually comparable;
# point estimates are identical to the paper, only CI width differs by <~0.005.
AUROC_BOOTSTRAP_N = 1000

# ---- The NEW judges --------------------------------------------------------
# family_of(key) classifies by prefix; a judge never foils against its own
# family. Revisions for the three REUSED models are copied verbatim from the
# frozen configs/models.yaml (bootstrap VERIFIES they still match — fail loud
# on drift). The two newly-generated Gemma sizes are not yet pinned: we load
# 'latest', RECORD the resolved commit into run_report.txt, and you pin it for
# camera-ready (exactly how the repo pinned its others; see PREREGISTRATION
# deviations log).
NEW_JUDGES = [
    # Tier 1 — promote existing foils to judges (text exists; only s2 is new)
    {"key": "llama-3.2-3b-instruct",     "hf_id": "meta-llama/Llama-3.2-3B-Instruct",
     "revision": "0cb88a4f764b7a12671c53f0838cd831a0843b95", "params_b": 3,
     "tier": 1, "needs_s1": False},
    {"key": "gemma-2-9b-it",             "hf_id": "google/gemma-2-9b-it",
     "revision": "11c9b309abf73637e4b6f9a3fa1e92e615547819", "params_b": 9,
     "tier": 1, "needs_s1": False},
    {"key": "mistral-7b-instruct-v0.3",  "hf_id": "mistralai/Mistral-7B-Instruct-v0.3",
     "revision": "c170c708c41dac9275d15a8fff4eca08d52bab71", "params_b": 7,
     "tier": 1, "needs_s1": False},
    # Tier 2 — new Gemma-2 sizes for the second scale curve (full generation)
    {"key": "gemma-2-2b-it",             "hf_id": "google/gemma-2-2b-it",
     "revision": None, "params_b": 2, "tier": 2, "needs_s1": True},
    {"key": "gemma-2-27b-it",            "hf_id": "google/gemma-2-27b-it",
     "revision": None, "params_b": 27, "tier": 2, "needs_s1": True, "shard": True},
]

# Per-GPU 4-bit shard budget for models that MUST split across both T4s.
# Consumed by src/utils.load_model_and_tokenizer via env MIRROR_MAX_MEMORY.
MAX_MEMORY_SHARDED = "13GiB,13GiB"

# ---- The common foil pool (family-exclusion) -------------------------------
FOIL_POOL_LLM = ["llama-3.2-3b-instruct", "gemma-2-9b-it", "mistral-7b-instruct-v0.3"]
HUMAN_FOIL    = "human"

# Qwen judges are reused as-is for the family curve + Holm family; params from
# the frozen config (bootstrap cross-checks).
QWEN_JUDGES = [
    ("qwen2.5-0.5b-instruct", 0.5), ("qwen2.5-1.5b-instruct", 1.5),
    ("qwen2.5-3b-instruct", 3), ("qwen2.5-7b-instruct", 7),
    ("qwen2.5-14b-instruct", 14),
]


def family_of(key: str) -> str:
    if key.startswith("qwen2.5"):
        return "qwen"
    if key.startswith("gemma-2"):
        return "gemma"
    if key.startswith("mistral"):
        return "mistral"
    if key.startswith("llama"):
        return "llama"
    if key == "human":
        return "human"
    return key


def foils_for(judge_key: str) -> list:
    """Common pool minus the judge's own family, plus human. All foil text
    pre-exists, so this never triggers new generation."""
    jf = family_of(judge_key)
    foils = [f for f in FOIL_POOL_LLM if family_of(f) != jf]
    return foils + [HUMAN_FOIL]


# ---- Pinned pip environment (T4-compatible) --------------------------------
# bitsandbytes 4-bit NF4 with float16 compute (T4 is Turing: no bf16). These
# pins are known-good on Kaggle's CUDA 12 image; loosen only if Kaggle's base
# image moves under you.
PIP_PACKAGES = [
    "transformers==4.44.2", "accelerate==0.34.2", "bitsandbytes==0.43.3",
    "datasets==2.21.0", "sentence-transformers==3.0.1", "scikit-learn==1.5.1",
    "pyyaml==6.0.2", "matplotlib==3.9.2",
]

if not (0 < len(NEW_JUDGES) == len({j["key"] for j in NEW_JUDGES})):
    raise ValueError("NEW_JUDGES keys must be unique and non-empty")

print("=" * 74)
print("MIRROR-TEST CROSS-FAMILY EXTENSION — run settings")
print("=" * 74)
print(f"  repo source    : clone {REPO_GIT_URL}@{REPO_GIT_REF}"
      f"{'  (OVERRIDE: ' + str(INPUT_ROOT_OVERRIDE) + ')' if INPUT_ROOT_OVERRIDE else ''}")
print(f"  working root   : {WORKING_ROOT}")
print(f"  checkpoints    : {CHECKPOINT_DIR}")
print(f"  outputs        : {OUTPUT_DIR}")
print(f"  HF cache       : {HF_CACHE_DIR}")
print(f"  budget (h)     : {BUDGET_HOURS}   DRY_RUN={DRY_RUN}  LOCAL_TEST={LOCAL_TEST}")
print(f"  enable 27B     : {ENABLE_27B}   (sharded {MAX_MEMORY_SHARDED} if on)")
print(f"  AUROC bootstrap: {AUROC_BOOTSTRAP_N}")
print("  new judges & family-excluded foils:")
for j in NEW_JUDGES:
    if j["key"] == "gemma-2-27b-it" and not ENABLE_27B:
        continue
    rev = (j["revision"] or "latest (will be recorded + flagged to pin)")
    print(f"    - {j['key']:26s} [{j['params_b']:>4}B, tier {j['tier']}, "
          f"fam={family_of(j['key'])}]  rev={rev[:16]}")
    print(f"        foils: {', '.join(foils_for(j['key']))}")
print("  reused Qwen judges: " + ", ".join(k for k, _ in QWEN_JUDGES))
print("=" * 74)


In [ ]:
# =========================== BOOTSTRAP ======================================
# Fail-loud environment setup. This cell NEVER resamples prompts or regenerates
# reused models: it only locates the frozen inputs, copies them into a writable
# tree, verifies their integrity, and wires up imports. If anything required is
# missing it raises immediately with a report of what was found vs expected —
# silently regenerating would break comparability with the published numbers.
# ===========================================================================

import os, sys, shutil, subprocess, importlib.util
from pathlib import Path

WORK = Path(WORKING_ROOT)
(WORK / "configs").mkdir(parents=True, exist_ok=True)
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


def _sh(cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


# ---- 1. pip (pinned; skipped in LOCAL_TEST) --------------------------------
if not LOCAL_TEST:
    try:
        _sh([sys.executable, "-m", "pip", "-q", "install", "-U", *PIP_PACKAGES])
    except subprocess.CalledProcessError as e:
        print(f"[pip] WARNING: pinned install returned {e.returncode}; "
              "continuing with whatever is already present.")

# ---- 2. obtain the repo: override / attached dataset -> git clone ----------
# Upload-and-run: by default we CLONE the pushed repo (code + frozen data +
# published results), so there is nothing to attach. If the repo is already
# attached as a Dataset or given via INPUT_ROOT_OVERRIDE, that is used instead.
# Frozen data is never regenerated.
def _has_repo(p: Path) -> bool:
    return (p / "mirror-test-llms" / "src" / "utils.py").exists()


def obtain_repo() -> Path:
    # (a) explicit override / LOCAL_TEST
    if INPUT_ROOT_OVERRIDE:
        p = Path(INPUT_ROOT_OVERRIDE)
        if not _has_repo(p) and (p / "src" / "utils.py").exists():
            p = p.parent            # user pointed straight at the repo dir
        if _has_repo(p):
            print(f"[repo] using INPUT_ROOT_OVERRIDE: {p}")
            return p
        raise SystemExit(f"[fatal] INPUT_ROOT_OVERRIDE has no mirror-test-llms/src: {p}")
    # (b) repo attached as a Kaggle Dataset (no clone needed)
    base = Path("/kaggle/input")
    if base.exists():
        cands = sorted({c.parent.parent.parent
                        for c in base.rglob("mirror-test-llms/src/utils.py")},
                       key=lambda x: len(str(x)))
        if cands:
            print(f"[repo] using attached dataset: {cands[0]}")
            return cands[0]
    # (c) git clone the pushed repo (the default upload-and-run path)
    clone_root = WORK / "_repo_src"
    if _has_repo(clone_root):
        print(f"[repo] reusing existing clone: {clone_root}")
        return clone_root
    url = REPO_GIT_URL
    tok = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if tok and url.startswith("https://github.com/"):
        url = url.replace("https://", f"https://{tok}@")   # private-repo auth
    print(f"[repo] cloning {REPO_GIT_URL}@{REPO_GIT_REF} -> {clone_root} "
          "(shallow; needs Internet = ON) ...", flush=True)
    # NB: run directly (not via _sh) so a token in `url` never prints to the log.
    rc = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_GIT_REF,
                         url, str(clone_root)]).returncode
    if rc != 0 or not _has_repo(clone_root):
        raise SystemExit(
            f"[fatal] could not clone {REPO_GIT_URL}@{REPO_GIT_REF}. Ensure the repo is "
            "pushed and public (or set a GITHUB_TOKEN secret for a private repo), "
            "Internet is ON, or attach the repo as a Dataset / set INPUT_ROOT_OVERRIDE.")
    return clone_root


INPUT_ROOT = obtain_repo()
REPO_DIR = INPUT_ROOT / "mirror-test-llms"
SRC_DIR = REPO_DIR / "src"
ORIG_CONFIG = REPO_DIR / "configs" / "models.yaml"
print(f"[repo] project dir: {REPO_DIR}")
for need in (SRC_DIR / "utils.py", ORIG_CONFIG, REPO_DIR / "data" / "prompts"):
    if not need.exists():
        raise FileNotFoundError(f"[repo] required path missing in repo: {need}")

# ---- 3. environment: writable root, HF cache, token ------------------------
os.environ["MIRROR_ROOT"] = str(WORK)
try:                                     # big weight cache off the 20 GB working cap
    Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["HF_HUB_CACHE"] = HF_CACHE_DIR
    print(f"[hf] weight cache: {HF_CACHE_DIR}")
except OSError:
    print(f"[hf] {HF_CACHE_DIR} not writable; using default ~/.cache/huggingface "
          "(fine for <=9B; gemma-2-27b-it ~54GB may need more room)")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("[auth] HF_TOKEN loaded from Kaggle secret")
    except Exception:
        print("[auth] no HF_TOKEN yet (gated Gemma-2/Llama will 403 until you add "
              "the Kaggle secret and accept the licenses)")

# ---- 4. copy frozen inputs into the writable tree (never clobber) ----------
def seed_tree(src_base: Path, subdirs, label: str) -> None:
    copied = skipped = 0
    for sub in subdirs:
        sdir = src_base / sub
        if not sdir.exists():
            continue
        for f in sdir.rglob("*"):
            if not f.is_file() or "__pycache__" in f.parts:
                continue
            dst = WORK / f.relative_to(src_base)
            if dst.exists():
                skipped += 1
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst)
            copied += 1
    print(f"[seed:{label}] copied {copied} files, kept {skipped} already-present")


# 4a. frozen data + published results from the repo dataset
seed_tree(REPO_DIR, ["data", "results"], "repo")
# 4b. any RE-ATTACHED prior session output (resume across sessions): other
#     /kaggle/input dirs carrying our working artifacts. Newer local wins.
if Path("/kaggle/input").exists():
    for cand in Path("/kaggle/input").iterdir():
        if cand.resolve() == INPUT_ROOT.resolve():
            continue
        if any((cand / s).exists() for s in
               ("data/generations", "results/judgments", "checkpoints",
                "extended_outputs")):
            seed_tree(cand, ["data", "results", "checkpoints", "extended_outputs"],
                      f"prior:{cand.name}")
            src_ck = cand / "checkpoints"
            if src_ck.exists():
                for f in src_ck.rglob("*"):
                    if f.is_file():
                        dst = Path(CHECKPOINT_DIR) / f.relative_to(src_ck)
                        if not dst.exists():
                            dst.parent.mkdir(parents=True, exist_ok=True)
                            shutil.copy2(f, dst)

# ---- 5. imports (utils + stats + numbered pipeline modules) ----------------
sys.path.insert(0, str(SRC_DIR))
import utils, stats_utils                                  # noqa: E402


def load_mod(name: str, filename: str):
    spec = importlib.util.spec_from_file_location(name, str(SRC_DIR / filename))
    m = importlib.util.module_from_spec(spec)
    sys.modules[name] = m
    spec.loader.exec_module(m)
    return m


mod_build = load_mod("mt_build_pairs", "02_build_pairs.py")
mod_base  = load_mod("mt_baselines", "05_baselines.py")
mod_stats = load_mod("mt_stats", "06_stats.py")
CFG = utils.load_config(str(ORIG_CONFIG))
print(f"[cfg] master seed={CFG['seed']}  domains={utils.DOMAINS}  "
      f"gen(temp={CFG['generation']['temperature']}, "
      f"seed_base={CFG['generation']['seed_base']}, "
      f"placebo_base={CFG['generation']['placebo_seed_base']})")

# ---- 6. verify frozen prompts (checksums) + required generations -----------
report = {"ok": [], "missing": [], "checksum_bad": []}
checks = (REPO_DIR / "data" / "prompts" / "CHECKSUMS.txt")
sums = {}
if checks.exists():
    for line in checks.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            h, name = line.split()[:2]
            sums[name] = h
for d in utils.DOMAINS:
    p = WORK / "data" / "prompts" / f"{d}.jsonl"
    if not p.exists():
        report["missing"].append(f"prompts/{d}.jsonl")
    elif d + ".jsonl" in sums and utils.sha256_file(p) != sums[d + ".jsonl"]:
        report["checksum_bad"].append(f"prompts/{d}.jsonl")
    else:
        report["ok"].append(f"prompts/{d}.jsonl")

N_TARGET = CFG["prompt_filters"]["n_per_domain"]
# Foil text that must ALREADY exist (never regenerated): every LLM foil in the
# pool, all domains, at least N_TARGET s1 records.
need_s1 = set(FOIL_POOL_LLM) | {j["key"] for j in NEW_JUDGES if not j["needs_s1"]}
for key in sorted(need_s1):
    for d in utils.DOMAINS:
        f = WORK / "data" / "generations" / f"{key}__{d}.jsonl"
        recs = utils.read_jsonl(f)
        n_s1 = sum(1 for r in recs if r.get("sample", "s1") == "s1")
        if n_s1 >= N_TARGET:
            report["ok"].append(f"gen/{key}__{d} (s1={n_s1})")
        else:
            report["missing"].append(f"gen/{key}__{d} (s1={n_s1}<{N_TARGET})")

qwen_judg = sorted((WORK / "results" / "judgments").glob("ppp__qwen2.5-*.jsonl"))
print("\n[startup] frozen-input verification")
print(f"    OK           : {len(report['ok'])} required files present")
print(f"    reused Qwen  : {len(qwen_judg)} judgment files "
      f"(needed for the Qwen curve + Holm family)")
if report["missing"] or report["checksum_bad"]:
    for m in report["missing"]:
        print(f"    MISSING      : {m}")
    for m in report["checksum_bad"]:
        print(f"    CHECKSUM BAD : {m}")
    raise SystemExit(
        "[fatal] required frozen inputs are missing or altered — refusing to "
        "proceed (regenerating them would break comparability with the paper). "
        "Attach the correct repo Dataset and rerun.")
if not qwen_judg:
    print("    [warn] no Qwen judgments found — the extension will still run, but "
          "scale_curves.png will show only the Gemma-2 curve and Holm will cover "
          "the new cells only.")

# ---- 7. write the patched config (adds new judges; foils=[human]) ----------
# WHY foils=[human]: model_index() lets later groups shadow earlier ones, so a
# key present in BOTH judges and foils resolves to group 'foils' and would make
# 01_generate treat it as a non-judge (no placebo s2). Keeping the promoted
# families out of `foils` guarantees is_judge=True so their s2 sample generates.
# Foil TEXT is still supplied explicitly via --foils to 02_build_pairs and read
# straight off disk, so nothing depends on the config foils list.
import copy, yaml                                          # noqa: E402
orig_foils = {m["key"]: m for m in CFG["foils"]}
drift = []
for j in NEW_JUDGES:
    if not j["needs_s1"] and j["key"] in orig_foils:
        want = orig_foils[j["key"]].get("revision")
        if want and j["revision"] and want != j["revision"]:
            drift.append((j["key"], j["revision"], want))
if drift:
    for k, mine, theirs in drift:
        print(f"    REVISION DRIFT {k}: config-cell={mine[:12]} vs frozen={theirs[:12]}")
    raise SystemExit("[fatal] reused-judge revision mismatch vs frozen config — "
                     "fix NEW_JUDGES revisions in the config cell to match.")

patched = copy.deepcopy(CFG)
patched.pop("_config_path", None)
existing_judge_keys = {m["key"] for m in patched["judges"]}
for j in NEW_JUDGES:
    if j["key"] == "gemma-2-27b-it" and not ENABLE_27B:
        continue
    if j["key"] in existing_judge_keys:
        continue
    patched["judges"].append({"key": j["key"], "hf_id": j["hf_id"],
                              "revision": j["revision"], "params_b": j["params_b"]})
patched["foils"] = [m for m in CFG["foils"] if m["key"] == "human"] or \
    [{"key": "human", "hf_id": None, "params_b": None}]
EXT_CONFIG = WORK / "configs" / "models_ext.yaml"
EXT_CONFIG.write_text(yaml.safe_dump(patched, sort_keys=False, allow_unicode=True),
                      encoding="utf-8")
print(f"\n[cfg] wrote patched config -> {EXT_CONFIG} "
      f"({len(patched['judges'])} judges, foils={[m['key'] for m in patched['foils']]})")

# ---- 8. GPU report + seeds + versions --------------------------------------
HAVE_GPU = False
try:
    import torch                                            # noqa: E402
    HAVE_GPU = torch.cuda.is_available()
    if HAVE_GPU:
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"[gpu] cuda:{i} {p.name} ({p.total_memory/2**30:.1f} GB)")
        if torch.cuda.device_count() >= 2:
            print("[gpu] 2 GPUs present -> gemma-2-27b-it can shard across both.")
    else:
        print("[gpu] no CUDA visible (Settings -> Accelerator -> GPU T4 x2).")
    torch.manual_seed(CFG["seed"])
except Exception as e:
    print(f"[gpu] torch unavailable ({e})")
import random as _random                                    # noqa: E402
_random.seed(CFG["seed"])
try:
    import numpy as _np; _np.random.seed(CFG["seed"])       # noqa: E402
except Exception:
    pass

HAVE_HF_TOKEN = bool(os.environ.get("HF_TOKEN"))
print("\n[versions]")
for name in ("transformers", "accelerate", "bitsandbytes", "datasets",
             "sklearn", "matplotlib", "torch"):
    try:
        print(f"    {name}: {importlib.import_module(name).__version__}")
    except Exception:
        print(f"    {name}: (not importable)")
print(f"[env] HAVE_GPU={HAVE_GPU}  HAVE_HF_TOKEN={HAVE_HF_TOKEN}  "
      f"MIRROR_ROOT={os.environ['MIRROR_ROOT']}")
print("[bootstrap] done.")


In [ ]:
# =========================== ORCHESTRATOR ===================================
# Runs the PUBLISHED src/ scripts (unmodified) as subprocesses, one model in
# memory at a time, with checkpoint/skip, a heartbeat, and a session time
# budget. Using the real scripts is what makes every new measurement
# byte-identical to the released Qwen cells. Correctness never depends on the
# done-checks: each script is internally resumable (skips finished items), so a
# wrong "not-done" verdict costs at most one model load.
#
# Per new judge J (foils P = family-excluded pool + human):
#   gen:J    01_generate.py   --models J --placebo        (only s2 is new for
#                                                           Tier-1 judges)
#   pairs:J  02_build_pairs.py --judges J --foils P...     (PPP + placebo + IPP)
#   ppp:J    03_judge_ppp.py   --judge J --include-placebo
#   ipp:J    04_judge_ipp.py   --judge J
#   ppl:J    05_baselines.py perplexity --judge J
#   stylo:J  05_baselines.py stylometric --judge J         (CPU)
# gemma-2-27b-it is scheduled LAST and sharded across both T4s.
# ===========================================================================

import gc, json, queue, subprocess, threading, time
from collections import deque
from pathlib import Path

T0 = time.monotonic()
STATE_PATH = Path(CHECKPOINT_DIR) / "orchestrator_state.json"
DOMAINS = utils.DOMAINS
N_TARGET = CFG["prompt_filters"]["n_per_domain"]
PLACEBO_N = CFG["generation"]["placebo_n_prompts"]
PY = [sys.executable]
EXT = ["--config", str(EXT_CONFIG)]
GEN, PAIRS, JUDG, BASE = (utils.GENERATIONS_DIR, utils.PAIRS_DIR,
                          utils.JUDGMENTS_DIR, utils.BASELINES_DIR)

ACTIVE_JUDGES = [j for j in NEW_JUDGES
                 if not (j["key"] == "gemma-2-27b-it" and not ENABLE_27B)]


def hours_left():
    return BUDGET_HOURS - (time.monotonic() - T0) / 3600


def _n(path):
    return sum(1 for ln in open(path, encoding="utf-8") if ln.strip()) \
        if Path(path).exists() else 0


# ------------------------------- done checks --------------------------------
def gen_done(key):
    for d in DOMAINS:
        recs = utils.read_jsonl(GEN / f"{key}__{d}.jsonl")
        s1 = sum(1 for r in recs if r.get("sample", "s1") == "s1")
        s2 = sum(1 for r in recs if r.get("sample") == "s2")
        if s1 < N_TARGET or s2 < min(PLACEBO_N, N_TARGET):
            return False
    return True


def pairs_done(key):
    P = foils_for(key)
    if not (PAIRS / f"ipp__{key}.jsonl").exists():
        return False
    for d in DOMAINS:
        if not (PAIRS / f"placebo__{key}__{d}.jsonl").exists():
            return False
        for f in P:
            if not (PAIRS / f"ppp__{key}__{f}__{d}.jsonl").exists():
                return False
    return True


def _expected_core_pairs(key):
    return sum(_n(PAIRS / f"ppp__{key}__{f}__{d}.jsonl")
               for f in foils_for(key) for d in DOMAINS)


def _expected_placebo_pairs(key):
    return sum(_n(PAIRS / f"placebo__{key}__{d}.jsonl") for d in DOMAINS)


def ppp_done(key):
    if not pairs_done(key):
        return False
    recs = utils.read_jsonl(JUDG / f"ppp__{key}.jsonl")
    core0 = sum(1 for r in recs if r["condition"] == "core" and r["phrasing"] == 0)
    plac = sum(1 for r in recs if r["condition"] == "placebo")
    return (core0 >= 2 * _expected_core_pairs(key)
            and plac >= 2 * _expected_placebo_pairs(key) and core0 > 0)


def ipp_done(key):
    exp = _n(PAIRS / f"ipp__{key}.jsonl")
    return exp > 0 and _n(JUDG / f"ipp__{key}.jsonl") >= exp


def ppl_done(key):
    core = sum(1 for r in utils.read_jsonl(BASE / f"ppl__{key}.jsonl")
               if r["condition"] == "core")
    return core > 0 and core >= _expected_core_pairs(key)


def stylo_done(key):
    if not pairs_done(key):
        return False
    for f in foils_for(key):
        for d in DOMAINS:
            pf = PAIRS / f"ppp__{key}__{f}__{d}.jsonl"
            if _n(pf) >= 20 and not (BASE / f"stylo__{key}__{f}__{d}.json").exists():
                return False
    return True


# ------------------------------- task table ---------------------------------
def T(tid, cmd, done_fn, after=(), gpu=True, gated=False, env_extra=None, est=90):
    return dict(id=tid, cmd=cmd, done_fn=done_fn, after=list(after), gpu=gpu,
                gated=gated, env_extra=env_extra or {}, est=est)


GEN_EST = {"llama-3.2-3b-instruct": 60, "gemma-2-9b-it": 90,
           "mistral-7b-instruct-v0.3": 70, "gemma-2-2b-it": 90,
           "gemma-2-27b-it": 200}
JUDGE_EST = {"llama-3.2-3b-instruct": 60, "gemma-2-9b-it": 110,
             "mistral-7b-instruct-v0.3": 90, "gemma-2-2b-it": 60,
             "gemma-2-27b-it": 240}
TASKS, PREFLIGHT = [], []
for j in ACTIVE_JUDGES:
    key = j["key"]
    P = foils_for(key)
    shard = bool(j.get("shard"))
    env_extra = {"MIRROR_MAX_MEMORY": MAX_MEMORY_SHARDED} if shard else {}
    gated = family_of(key) in ("gemma", "llama")
    TASKS.append(T(f"gen:{key}", PY + [str(SRC_DIR / "01_generate.py")] + EXT
                   + ["--models", key, "--placebo"],
                   (lambda k=key: gen_done(k)), gated=gated, env_extra=env_extra,
                   est=GEN_EST.get(key, 120)))
    TASKS.append(T(f"pairs:{key}", PY + [str(SRC_DIR / "02_build_pairs.py")] + EXT
                   + ["--judges", key, "--foils", *P, "--domains", *DOMAINS],
                   (lambda k=key: pairs_done(k)), after=[f"gen:{key}"], gpu=False,
                   est=8))
    TASKS.append(T(f"ppp:{key}", PY + [str(SRC_DIR / "03_judge_ppp.py")] + EXT
                   + ["--judge", key, "--include-placebo"],
                   (lambda k=key: ppp_done(k)), after=[f"pairs:{key}"],
                   gated=gated, env_extra=env_extra, est=JUDGE_EST.get(key, 120)))
    if RUN_IPP:
        TASKS.append(T(f"ipp:{key}", PY + [str(SRC_DIR / "04_judge_ipp.py")] + EXT
                       + ["--judge", key],
                       (lambda k=key: ipp_done(k)), after=[f"pairs:{key}"],
                       gated=gated, env_extra=env_extra, est=25))
    TASKS.append(T(f"ppl:{key}", PY + [str(SRC_DIR / "05_baselines.py"), "perplexity"]
                   + EXT + ["--judge", key],
                   (lambda k=key: ppl_done(k)), after=[f"pairs:{key}"],
                   gated=gated, env_extra=env_extra, est=JUDGE_EST.get(key, 120) // 2))
    if RUN_STYLOMETRY:
        TASKS.append(T(f"stylo:{key}", PY + [str(SRC_DIR / "05_baselines.py"),
                       "stylometric"] + EXT + ["--judge", key],
                       (lambda k=key: stylo_done(k)), after=[f"pairs:{key}"],
                       gpu=False, est=8))

# Order: Tier 1 (cheap, existing text) -> Tier 2 2B -> 27B LAST. Within a judge
# the `after` deps already serialise gen->pairs->{ppp,ipp,ppl,stylo}.
_order = {"llama-3.2-3b-instruct": 0, "mistral-7b-instruct-v0.3": 1,
          "gemma-2-9b-it": 2, "gemma-2-2b-it": 3, "gemma-2-27b-it": 9}
TASKS.sort(key=lambda t: (_order.get(t["id"].split(":", 1)[1], 5),
                          t["id"].startswith(("ppp", "ipp", "ppl", "stylo"))))


# ------------------------- subprocess runner --------------------------------
STATUS, HEARTBEAT_S = {}, 60


def clock():
    el = int(time.monotonic() - T0)
    return f"[{el//3600}:{el%3600//60:02d}]"


def save_state():
    STATE_PATH.parent.mkdir(parents=True, exist_ok=True)
    STATE_PATH.write_text(json.dumps(
        {"budget_hours": BUDGET_HOURS, "tasks": STATUS}, indent=2), encoding="utf-8")


def run_task(t, seq, n):
    tail = deque(maxlen=80)
    env = dict(os.environ)
    env.update(t["env_extra"])
    if t["env_extra"]:
        print(f"{clock()} [env] {t['env_extra']}", flush=True)
    print(f"\n{clock()} ===== {seq}/{n}: {t['id']} (est ~{t['est']}m | "
          f"{hours_left():.1f}h left) =====\n{clock()} $ {' '.join(t['cmd'])}", flush=True)
    start = time.monotonic()
    proc = subprocess.Popen(t["cmd"], cwd=str(REPO_DIR), env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    q = queue.Queue()
    threading.Thread(target=lambda: ([q.put(ln) for ln in proc.stdout], q.put(None)),
                     daemon=True).start()
    killed = eof = False
    last = time.monotonic()
    while not eof:
        try:
            ln = q.get(timeout=HEARTBEAT_S)
        except queue.Empty:
            print(f"{clock()} [heartbeat] {t['id']} up "
                  f"{(time.monotonic()-start)/60:.0f}m, quiet "
                  f"{(time.monotonic()-last)/60:.0f}m (downloads/loading are silent)",
                  flush=True)
        else:
            if ln is None:
                eof = True
            else:
                print(ln, end="", flush=True)
                tail.append(ln.rstrip())
                last = time.monotonic()
        if hours_left() <= 0 and not killed:
            print(f"\n{clock()} [budget] time up — pausing {t['id']} "
                  "(resumes next session)", flush=True)
            proc.terminate()
            killed = True
    rc = proc.wait()
    STATUS[t["id"]]["tail"] = list(tail)[-15:]
    STATUS[t["id"]]["minutes"] = round((time.monotonic() - start) / 60, 1)
    if killed:
        return "paused"
    if rc != 0:
        low = "\n".join(tail).lower()
        if any(x in low for x in ("401", "403", "gated", "unauthorized", "restricted")):
            return "auth-error"
        if "out of memory" in low or "cuda oom" in low:
            return "oom"
        return "failed"
    return "ran" if t["done_fn"]() else "partial"


# --------------------- 27B sharded pre-flight (verify NLL) ------------------
def preflight_27b():
    marker = Path(CHECKPOINT_DIR) / "preflight_27b.json"
    if marker.exists():
        print(f"{clock()} [preflight] 27B already verified: "
              f"{marker.read_text(encoding='utf-8')[:160]}")
        return json.loads(marker.read_text(encoding="utf-8"))
    j = next(x for x in ACTIVE_JUDGES if x["key"] == "gemma-2-27b-it")
    os.environ["MIRROR_MAX_MEMORY"] = MAX_MEMORY_SHARDED
    print(f"{clock()} [preflight] loading gemma-2-27b-it sharded "
          f"({MAX_MEMORY_SHARDED}); first run downloads ~54GB — this is slow.")
    import torch
    model, tok = utils.load_model_and_tokenizer(j["hf_id"], j.get("revision"))
    ctx = utils.build_chat_text(tok, user="Write one short sentence about the sea.",
                                system=None)
    gen = utils.sampled_generate(model, tok, ctx, temperature=CFG["generation"]["temperature"],
                                 top_p=CFG["generation"]["top_p"], max_new_tokens=16,
                                 seed=CFG["generation"]["seed_base"])
    nll, ntok = utils.continuation_mean_nll(model, tok, ctx, gen or "The sea is vast.")
    devs = sorted({str(p.device) for p in model.parameters()})
    ok = (isinstance(nll, float) and nll == nll and nll != float("inf") and ntok > 0
          and len(devs) >= 1)
    res = {"gen_ok": bool(gen), "nll": round(float(nll), 4), "n_tok": int(ntok),
           "param_devices": devs, "resolved_revision": utils.resolved_revision(model),
           "verified": bool(ok)}
    del model
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    marker.write_text(json.dumps(res, indent=2), encoding="utf-8")
    print(f"{clock()} [preflight] {res}")
    if not ok:
        raise SystemExit("[fatal] 27B sharded generation/NLL sanity FAILED — "
                         "aborting 27B (set ENABLE_27B=False to ship 2B/9B curve).")
    return res


# ------------------------------- run loop -----------------------------------
print(f"\n{clock()} evaluating what is already done ...")
runnable = []
print(f"\n{'TASK':<32}{'STATE':<10}{'EST':>5}")
for t in TASKS:
    done = False
    try:
        done = t["done_fn"]()
    except Exception:
        done = False
    STATUS[t["id"]] = {"status": "done" if done else "todo"}
    if not done:
        runnable.append(t)
    print(f"{t['id']:<32}{STATUS[t['id']]['status']:<10}{t['est']:>4}m")
print(f"\n{clock()} {len(TASKS)-len(runnable)}/{len(TASKS)} already done; "
      f"~{sum(t['est'] for t in runnable)/60:.1f}h of work remains; budget {BUDGET_HOURS}h")
save_state()

# 27B preflight before any 27B GPU task (once), unless skipping GPU.
_want_27b = ENABLE_27B and any(t["id"].startswith(("gen:gemma-2-27b", "ppp:gemma-2-27b",
                               "ipp:gemma-2-27b", "ppl:gemma-2-27b")) for t in runnable)
if _want_27b and HAVE_GPU and HAVE_HF_TOKEN and not DRY_RUN:
    try:
        preflight_27b()
    except SystemExit as e:
        print(e)
        ACTIVE_JUDGES = [x for x in ACTIVE_JUDGES if x["key"] != "gemma-2-27b-it"]
        runnable = [t for t in runnable if "gemma-2-27b" not in t["id"]]

seq = 0
for t in runnable:
    st = STATUS[t["id"]]
    deps = [d for d in t["after"] if STATUS.get(d, {}).get("status") not in ("done", "ran")]
    if deps:
        st.update(status="blocked", note=f"waiting: {','.join(deps)}")
        print(f"{clock()} [skip] {t['id']}: blocked by {','.join(deps)}")
    elif DRY_RUN:
        st.update(status="would-run")
        print(f"{clock()} [dry-run] would run {t['id']} (~{t['est']}m)")
    elif t["gpu"] and not HAVE_GPU:
        st.update(status="skipped", note="no GPU")
        print(f"{clock()} [skip] {t['id']}: no GPU this session")
    elif t["gated"] and not HAVE_HF_TOKEN:
        st.update(status="skipped", note="needs HF_TOKEN + accepted license")
        print(f"{clock()} [skip] {t['id']}: needs HF_TOKEN secret + license")
    elif hours_left() <= 0:
        st.update(status="deferred", note="budget spent")
        print(f"{clock()} [defer] {t['id']}: budget spent")
    else:
        seq += 1
        outcome = run_task(t, seq, len(runnable))
        st.update(status=outcome)
        print(f"{clock()} [task] {t['id']} -> {outcome.upper()} "
              f"({st.get('minutes','?')}m vs est {t['est']}m)")
    save_state()

_c = {}
for v in STATUS.values():
    _c[v["status"]] = _c.get(v["status"], 0) + 1
print(f"\n{clock()} [orchestrator] pass complete — "
      + " ".join(f"{k}:{v}" for k, v in sorted(_c.items()))
      + f" | {hours_left():.1f}h budget left")
print("Rerun this notebook (fresh session, re-attach prior output) to continue "
      "any 'paused'/'deferred'/'skipped' tasks — completed units are skipped.")


In [ ]:
# ===================== EXTENDED STATS + OUTPUTS =============================
# Recomputes every cell (reused Qwen + new families) from the raw judgment
# files, using the PUBLISHED statistics functions unchanged, so the extended
# table is directly comparable to the paper's Table 1. Holm is recomputed over
# ALL cells present (not hardcoded to the old 20). Writes the task deliverables
# to /kaggle/working/.
#
# Fidelity notes:
#  * cell_stats() is reused verbatim — including its 500-resample AUROC CI and
#    the config's 1000-resample kappa CI — so numbers match the released ones
#    exactly rather than "close".
#  * kappa is reported but is known to saturate toward 0 when either party is
#    near-constant (the paper concedes this). The dissociation is therefore
#    carried by (a) the implicit-vs-explicit ACCURACY GAP and (b) the
#    non-saturating margin-margin Spearman rho (tools/margin_correlation.py).
# ===========================================================================

import csv as _csv
from pathlib import Path
from collections import defaultdict

alpha = CFG["stats"]["alpha"]
KAPPA_BOOT = CFG["stats"]["bootstrap_n"]          # 1000, per frozen config
OUT = Path(OUTPUT_DIR)
(OUT / "raw").mkdir(parents=True, exist_ok=True)

LLM_FOILS_ALL = ["llama-3.2-3b-instruct", "gemma-2-9b-it", "mistral-7b-instruct-v0.3"]
QWEN_FOILS = LLM_FOILS_ALL + ["human"]            # original study foils


def judge_foils(judge_key):
    return QWEN_FOILS if family_of(judge_key) == "qwen" else foils_for(judge_key)


# Params for every judge (Qwen from frozen config; new from the config cell).
PARAMS = {k: v for k, v in QWEN_JUDGES}
PARAMS.update({j["key"]: j["params_b"] for j in NEW_JUDGES})

# Judges of interest = the instruct scale families only. Base checkpoints
# (keys ending in '-base') are a separate mechanism ablation the paper keeps in
# base_vs_instruct.csv, NOT in Table 1 — excluding them keeps the Holm family
# and the scale curve exactly comparable to the published main table.
JUDGES_OF_INTEREST = [k for k, _ in QWEN_JUDGES] + [j["key"] for j in NEW_JUDGES]

# All judges that actually have judgments on disk (partial sessions welcome).
runs = mod_stats.load_ppp_runs()
items = mod_stats.items_from_runs(runs)
runs_judges = {r["judge"] for r in runs}
present_judges = [k for k in JUDGES_OF_INTEREST if k in runs_judges]
stylo_sum, stylo_pairs = mod_stats.load_stylo()
ppl = mod_stats.load_ppl()
print(f"[stats] {len(runs)} runs loaded; judges present: {present_judges}")

# --------------------------- extended_table1 --------------------------------
main_rows, pvals = [], []
cell_n = []
for judge in present_judges:
    for foil in judge_foils(judge):
        st = mod_stats.cell_stats(items, judge, foil, alpha=alpha)
        if st is None:
            continue
        st["family"] = family_of(judge)
        st["params_b"] = PARAMS.get(judge, "")
        pvals.append((f"{judge}|{foil}", st["p_binomial"]))
        # stylometric pairwise acc pooled across domains by item count
        accs, ns = [], []
        for (j, f, d), s in stylo_sum.items():
            if j == judge and f == foil:
                accs.append(s["acc_pairwise_char_ngram"] * s["n_pairs"])
                ns.append(s["n_pairs"])
        st["stylo_pair_acc"] = (sum(accs) / sum(ns)) if ns else ""
        kp = mod_stats.judge_vs_ppl_kappa(items, judge, foil, ppl, KAPPA_BOOT)
        if kp:
            st.update({"ppl_rule_acc": kp["ppl_rule_acc"], "kappa": kp["kappa"],
                       "kappa_ci_lo": kp["kappa_ci_lo"], "kappa_ci_hi": kp["kappa_ci_hi"]})
        else:
            st.update({"ppl_rule_acc": "", "kappa": "", "kappa_ci_lo": "", "kappa_ci_hi": ""})
        main_rows.append(st)
        cell_n.append(st["n_items"])

# Holm across the FULL family of judge x foil cells now present
holm = {h["name"]: h for h in stats_utils.holm_bonferroni(pvals, alpha)}
for r in main_rows:
    h = holm.get(f"{r['judge']}|{r['foil']}")
    r["p_holm"] = h["p_holm"] if h else ""
    r["reject_holm"] = h["reject"] if h else ""

TABLE1_COLS = ["family", "judge", "params_b", "foil", "n_items", "acc", "ci_lo", "ci_hi",
               "auroc", "auroc_ci_lo", "auroc_ci_hi", "stylo_pair_acc", "ppl_rule_acc",
               "kappa", "kappa_ci_lo", "kappa_ci_hi", "consistency", "n_decisive",
               "k_decisive", "p_binomial", "p_holm", "reject_holm"]


def _write_csv(path, cols, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = _csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({c: (f"{r[c]:.4f}" if isinstance(r.get(c), float) else r.get(c, ""))
                        for c in cols})
    print(f"[out] {path}  ({len(rows)} rows)")


_write_csv(OUT / "extended_table1.csv", TABLE1_COLS, main_rows)
HOLM_FAMILY_SIZE = len(pvals)
print(f"[stats] Holm family recomputed over {HOLM_FAMILY_SIZE} judge x foil cells "
      f"(paper's original was 20).")

# ------------------- pooled-over-LLM-foils helpers --------------------------
def pooled_llm_ppp(judge):
    """Explicit accuracy pooled over the judge's LLM foils (human excluded) —
    the paper's headline definition. Returns (acc, lo, hi, n)."""
    llm = [f for f in judge_foils(judge) if f != "human"]
    scores = [it["score"] for (j, f, d, c, ph, pid), it in items.items()
              if j == judge and f in llm and c == "core" and ph == 0 and it["score"] is not None]
    if not scores:
        return None
    acc = stats_utils.mean(scores)
    lo, hi = stats_utils.wilson_ci(acc, len(scores), alpha)
    return acc, lo, hi, len(scores)


def pooled_llm_ppl(judge, llm_only=True):
    """Implicit (lower-perplexity rule) accuracy. llm_only pools over the same
    LLM foils as the explicit headline (apples-to-apples on one item universe);
    llm_only=False pools over ALL foils incl. human (reproduces the paper's
    prose headline, e.g. 0.795 for the 0.5B)."""
    foils = [f for f in judge_foils(judge)]
    if llm_only:
        foils = [f for f in foils if f != "human"]
    hits = [1.0 if r["rule_chose_self"] else 0.0
            for r in utils.read_jsonl(utils.BASELINES_DIR / f"ppl__{judge}.jsonl")
            if r["condition"] == "core" and r["foil"] in foils]
    return (stats_utils.mean(hits), len(hits)) if hits else None


def margin_rho(judge):
    """Non-saturating companion to kappa (tools/margin_correlation.py): Spearman
    rho between the judge's position-debiased letter-preference margin and the
    perplexity rule's margin, over shared core pairs. Unlike kappa it does not
    collapse when a party is near-constant; larger |rho| means the graded verbal
    margin tracks likelihood more. Reported with a bootstrap CI per judge — the
    DISSOCIATION itself is carried by the implicit-minus-explicit accuracy gap,
    not by assuming rho is ~0 (in Qwen rho is small and rises with scale)."""
    jr = [r for r in utils.read_jsonl(utils.JUDGMENTS_DIR / f"ppp__{judge}.jsonl")
          if r["condition"] == "core" and r["phrasing"] == 0]
    by_pair = defaultdict(dict)
    for r in jr:
        by_pair[r["pair_id"]][r["order"]] = r["logp_A"] - r["logp_B"]
    rule = {r["pair_id"]: r["nll_foil"] - r["nll_self"]
            for r in utils.read_jsonl(utils.BASELINES_DIR / f"ppl__{judge}.jsonl")
            if r["condition"] == "core"}
    pts = [((d["self_A"] - d["self_B"]) / 2, rule[p]) for p, d in by_pair.items()
           if "self_A" in d and "self_B" in d and p in rule]
    if len(pts) < 10:
        return None
    rho = stats_utils.spearman_rho([a for a, _ in pts], [b for _, b in pts])
    lo, hi = stats_utils.bootstrap_ci(
        lambda s: stats_utils.spearman_rho([a for a, _ in s], [b for _, b in s]),
        pts, n_boot=KAPPA_BOOT)
    return rho, lo, hi, len(pts)


# --------------------------- dissociation_summary ---------------------------
disso_rows = []
for judge in present_judges:
    ex = pooled_llm_ppp(judge)
    im = pooled_llm_ppl(judge)
    if ex is None or im is None:
        continue
    kaps = [r["kappa"] for r in main_rows
            if r["judge"] == judge and r["foil"] != "human" and isinstance(r.get("kappa"), float)]
    hum = mod_stats.cell_stats(items, judge, "human", alpha=alpha)
    mr = margin_rho(judge)
    im_all = pooled_llm_ppl(judge, llm_only=False)
    row = {
        "family": family_of(judge), "judge": judge, "params_b": PARAMS.get(judge, ""),
        "n_llm_pairs": ex[3],
        "implicit_ppl_acc": im[0],
        "implicit_ppl_acc_allfoils": (im_all[0] if im_all else ""),
        "explicit_ppp_acc": ex[0],
        "explicit_ci_lo": ex[1], "explicit_ci_hi": ex[2],
        "acc_gap_implicit_minus_explicit": im[0] - ex[0],
        "kappa_median": (sorted(kaps)[len(kaps) // 2] if kaps else ""),
        "kappa_absmax": (max(abs(k) for k in kaps) if kaps else ""),
        "margin_rho": (mr[0] if mr else ""), "margin_rho_lo": (mr[1] if mr else ""),
        "margin_rho_hi": (mr[2] if mr else ""),
        "auroc_human": (hum["auroc"] if hum else ""),
        "auroc_human_lo": (hum["auroc_ci_lo"] if hum else ""),
        "auroc_human_hi": (hum["auroc_ci_hi"] if hum else ""),
    }
    disso_rows.append(row)

DISSO_COLS = ["family", "judge", "params_b", "n_llm_pairs", "implicit_ppl_acc",
              "implicit_ppl_acc_allfoils",
              "explicit_ppp_acc", "explicit_ci_lo", "explicit_ci_hi",
              "acc_gap_implicit_minus_explicit", "kappa_median", "kappa_absmax",
              "margin_rho", "margin_rho_lo", "margin_rho_hi",
              "auroc_human", "auroc_human_lo", "auroc_human_hi"]
disso_rows.sort(key=lambda r: (r["family"], r["params_b"] or 0))
_write_csv(OUT / "dissociation_summary.csv", DISSO_COLS, disso_rows)

# ------------------------------ raw per-cell --------------------------------
# Copy the NEW judges' raw judgments/baselines and write per-(judge,foil)
# JSON + parquet-if-available for reproducibility.
try:
    import pandas as _pd
    _HAVE_PD = True
except Exception:
    _HAVE_PD = False
runs_by_cell = defaultdict(list)
for r in runs:
    runs_by_cell[(r["judge"], r["foil"])].append(r)
new_keys = {j["key"] for j in NEW_JUDGES}
n_raw = 0
for (judge, foil), rows in runs_by_cell.items():
    if judge not in new_keys:
        continue
    stem = OUT / "raw" / f"ppp__{judge}__{foil}"
    utils.write_jsonl(str(stem) + ".jsonl", rows)
    if _HAVE_PD:
        try:
            _pd.DataFrame(rows).to_parquet(str(stem) + ".parquet", index=False)
        except Exception:
            pass
    n_raw += 1
for judge in new_keys:
    for src in (utils.BASELINES_DIR / f"ppl__{judge}.jsonl",
                utils.JUDGMENTS_DIR / f"ipp__{judge}.jsonl"):
        if src.exists():
            (OUT / "raw" / src.name).write_text(src.read_text(encoding="utf-8"),
                                                encoding="utf-8")
print(f"[out] raw per-cell judgments -> {OUT / 'raw'} ({n_raw} PPP cells"
      f"{' + parquet' if _HAVE_PD else ', JSONL only'})")

# Stash for the figure + finalize cells.
CELL_N = cell_n
EXT_MAIN_ROWS = main_rows
EXT_DISSO_ROWS = disso_rows
print("[stats] extended tables written.")


In [ ]:
# ========================= SCALE CURVES FIGURE =============================
# Qwen2.5 vs Gemma-2, three panels vs log judge size:
#   (1) self-recognition PPP accuracy (pooled over LLM foils, human excluded)
#   (2) lower-perplexity-rule accuracy (implicit self-information)
#   (3) human-vs-machine AUROC
# Two family series only, so identity is carried by color AND marker AND
# linestyle (CVD-safe blue-circle-solid vs amber-triangle-dashdot; grayscale
# values 119 vs 174) with a legend always present — never color alone. Chance
# line at 0.5. Reuses the repo's axis chrome for visual identity with the
# paper's Figure 1.
# ===========================================================================

from pathlib import Path

FAMILY_STYLE = {                       # (hex, marker, linestyle, label)
    "qwen":  ("#2a78d6", "o", "-",  "Qwen2.5"),
    "gemma": ("#eda100", "^", "-.", "Gemma-2"),
}
PANELS = [
    ("explicit_ppp_acc", ("explicit_ci_lo", "explicit_ci_hi"),
     "Self-recognition (PPP acc)"),
    ("implicit_ppl_acc", None, "Perplexity-rule acc"),
    ("auroc_human", ("auroc_human_lo", "auroc_human_hi"),
     "Human-vs-machine AUROC"),
]


def _wilson_band(acc, n):
    lo, hi = stats_utils.wilson_ci(acc, n, CFG["stats"]["alpha"])
    return lo, hi


def build_scale_curves(disso_rows):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception as e:
        print(f"[figure] matplotlib unavailable ({e}); skipping scale_curves.png")
        return None

    by_family = {}
    for r in disso_rows:
        by_family.setdefault(r["family"], []).append(r)
    for fam in by_family:
        by_family[fam].sort(key=lambda r: r["params_b"]
                            if isinstance(r["params_b"], (int, float)) else 0.0)
    fams = [f for f in ("qwen", "gemma") if by_family.get(f)]
    if not fams:
        print("[figure] no family rows yet; skipping scale_curves.png")
        return None

    plt.rcParams.update({"font.size": 8, "font.family": "DejaVu Sans",
                         "text.color": mod_stats.INK, "axes.labelcolor": mod_stats.INK})
    fig, axes = plt.subplots(1, 3, figsize=(9.2, 2.8), constrained_layout=True)
    all_sizes = sorted({r["params_b"] for rows in by_family.values() for r in rows})

    for panel, (metric, ci, title) in enumerate(PANELS):
        ax = axes[panel]
        for fam in fams:
            color, marker, ls, label = FAMILY_STYLE[fam]
            rows = [r for r in by_family[fam]
                    if isinstance(r.get(metric), float)
                    and (r["params_b"] not in (None, ""))]
            if not rows:
                continue
            xs = [r["params_b"] for r in rows]
            ys = [r[metric] for r in rows]
            if ci is None:                       # implicit acc: Wilson from n
                bands = [_wilson_band(r[metric], r["n_llm_pairs"]) for r in rows]
                lo = [max(0.0, y - b[0]) for y, b in zip(ys, bands)]
                hi = [max(0.0, b[1] - y) for y, b in zip(ys, bands)]
            else:
                lo = [max(0.0, r[metric] - r[ci[0]]) for r in rows
                      if isinstance(r.get(ci[0]), float)]
                hi = [max(0.0, r[ci[1]] - r[metric]) for r in rows
                      if isinstance(r.get(ci[1]), float)]
                if len(lo) != len(ys):
                    lo = hi = None
            ax.errorbar(xs, ys, yerr=([lo, hi] if lo is not None else None),
                        color=color, marker=marker, linestyle=(ls if len(xs) > 1 else "None"),
                        linewidth=1.5, markersize=5, markeredgecolor=mod_stats.INK,
                        markeredgewidth=0.4, capsize=2, elinewidth=0.8, label=label)
        ax.axhline(0.5, color=mod_stats.MUTED, linewidth=0.8, linestyle=(0, (2, 2)))
        ax.text(ax.get_xlim()[1], 0.505, "chance", ha="right", va="bottom",
                fontsize=6, color=mod_stats.MUTED)
        ax.set_xscale("log")
        ax.set_xticks(all_sizes)
        ax.set_xticklabels([f"{s:g}" for s in all_sizes])
        ax.minorticks_off()
        ax.set_xlabel("Judge parameters (B, log scale)")
        ax.set_title(title, fontsize=8, color=mod_stats.INK)
        ax.set_ylim(0.3, 1.0)
        mod_stats._style_axes(ax)
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="lower center", ncol=len(labels),
                   bbox_to_anchor=(0.5, -0.08), frameon=False, fontsize=8)

    targets = [Path(WORKING_ROOT) / "scale_curves.png",
               Path(OUTPUT_DIR) / "scale_curves.png"]
    fig_dir = utils.FIGURES_DIR
    fig_dir.mkdir(parents=True, exist_ok=True)
    targets.append(fig_dir / "scale_curves_extended.png")
    for t in targets:
        fig.savefig(t, dpi=300, bbox_inches="tight")
    fig.savefig(fig_dir / "scale_curves_extended.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"[figure] scale_curves.png -> {targets[0]} (families: {', '.join(fams)})")
    return str(targets[0])


SCALE_CURVE_PATH = build_scale_curves(EXT_DISSO_ROWS)


In [ ]:
# ============================ FINALIZE ======================================
# Writes run_report.txt (provenance a reviewer needs) and prints the closing
# summary. Safe to run repeatedly; reflects whatever has completed so far.
# ===========================================================================

import datetime, json
from pathlib import Path

WORK = Path(WORKING_ROOT)
lines = []


def out(s=""):
    lines.append(s)
    print(s)


status = globals().get("STATUS", {})
disso = globals().get("EXT_DISSO_ROWS", [])
cell_n = globals().get("CELL_N", [])
holm_n = globals().get("HOLM_FAMILY_SIZE", "n/a")

out("=" * 74)
out("MIRROR-TEST CROSS-FAMILY EXTENSION — run report")
out(f"generated: {datetime.datetime.now().isoformat(timespec='seconds')}")
out("=" * 74)

# ---- task outcomes + timings ----------------------------------------------
gpu_min = cpu_min = 0.0
outcomes = {}
out("\n[tasks]")
for tid, s in status.items():
    st = s.get("status", "?")
    outcomes[st] = outcomes.get(st, 0) + 1
    mins = s.get("minutes")
    if mins:
        if tid.split(":")[0] in ("gen", "ppp", "ipp", "ppl"):
            gpu_min += mins
        else:
            cpu_min += mins
    out(f"    {tid:<30} {st:<12} {('%.0fm' % mins) if mins else ''}")
out(f"\n    summary: " + " ".join(f"{k}={v}" for k, v in sorted(outcomes.items())))
out(f"    GPU time this session ~= {gpu_min/60:.2f} h ; CPU ~= {cpu_min/60:.2f} h")

# ---- reused vs generated ---------------------------------------------------
out("\n[data provenance] (frozen inputs are never regenerated)")
new_keys = [j["key"] for j in NEW_JUDGES]
for key in new_keys:
    s1 = s2 = 0
    rev = None
    for d in utils.DOMAINS:
        for r in utils.read_jsonl(utils.GENERATIONS_DIR / f"{key}__{d}.jsonl"):
            if r.get("sample", "s1") == "s1":
                s1 += 1
            elif r.get("sample") == "s2":
                s2 += 1
            rev = rev or r.get("revision")
    is_reuse = not next(j["needs_s1"] for j in NEW_JUDGES if j["key"] == key)
    tag = "s1 REUSED (frozen), s2 generated" if is_reuse else "s1+s2 generated"
    out(f"    {key:<26} s1={s1:<4} s2={s2:<4} [{tag}]")
    out(f"        resolved revision: {rev or '(not generated yet)'}")
out("    reused unchanged: all Qwen2.5 generations/pairs/judgments; all foil s1 text.")

# ---- revisions to pin for camera-ready ------------------------------------
out("\n[pin these revisions in configs/models.yaml for camera-ready]")
for key in ("gemma-2-2b-it", "gemma-2-27b-it"):
    rev = None
    for d in utils.DOMAINS:
        recs = utils.read_jsonl(utils.GENERATIONS_DIR / f"{key}__{d}.jsonl")
        if recs:
            rev = recs[0].get("revision")
            break
    out(f"    {key}: {rev or '(not generated this session)'}")
pf = Path(CHECKPOINT_DIR) / "preflight_27b.json"
if pf.exists():
    out(f"\n[27B sharded pre-flight] {pf.read_text(encoding='utf-8')}")

# ---- statistics provenance + power ----------------------------------------
out("\n[statistics]")
out(f"    Holm–Bonferroni family recomputed over ALL {holm_n} judge x foil cells "
    f"present (paper's original: 20). Significance flags in extended_table1.csv "
    f"reflect this larger family.")
out(f"    AUROC 95% CI: {500} bootstrap resamples (repo cell_stats default, kept for "
    f"exact comparability with the paper); kappa CI: {CFG['stats']['bootstrap_n']} "
    f"resamples (frozen config).")
if cell_n:
    sn = sorted(cell_n)
    med = sn[len(sn) // 2]
    out(f"    per-cell n: min={min(sn)} median={med} max={max(sn)} across "
        f"{len(sn)} cells.")
    try:
        p80 = None
        for pa in [x / 1000 for x in range(505, 800)]:
            if stats_utils.power_exact_binomial(med, pa) >= 0.80:
                p80 = pa
                break
        out(f"    power note: exact binomial, alpha={CFG['stats']['alpha']}, at the "
            f"median decisive-item count the smallest true accuracy detectable at "
            f"80% power is ~{p80:.3f}; smaller effects may be missed (per-cell CIs "
            f"in extended_table1.csv show the actual precision).")
    except Exception:
        pass

# ---- dissociation headline -------------------------------------------------
if disso:
    out("\n[dissociation replication] implicit (PPL-rule) vs explicit (PPP), per judge")
    out(f"    {'judge':<26}{'impl':>6}{'expl':>7}{'gap':>7}{'margin_rho':>12}")
    for r in disso:
        mr = f"{r['margin_rho']:+.3f}" if isinstance(r.get("margin_rho"), float) else "  n/a"
        out(f"    {r['judge']:<26}{r['implicit_ppl_acc']:>6.3f}"
            f"{r['explicit_ppp_acc']:>7.3f}{r['acc_gap_implicit_minus_explicit']:>7.3f}"
            f"{mr:>12}")
    out("    The large implicit–explicit ACCURACY GAP is the dissociation: it is "
        "robust and does NOT saturate like kappa. margin_rho is a continuous "
        "companion (in Qwen it is small and rises modestly with scale, ~0 to ~0.34) "
        "— report it per judge rather than claiming rho~0.")

# ---- outputs written -------------------------------------------------------
out("\n[outputs in /kaggle/working/]")
for rel in ("extended_table1.csv", "scale_curves.png",
            "extended_outputs/dissociation_summary.csv",
            "extended_outputs/extended_table1.csv", "extended_outputs/raw"):
    p = WORK / rel
    if p.exists():
        out(f"    OK  {rel}")
disso_top = WORK / "dissociation_summary.csv"
if not disso_top.exists() and (Path(OUTPUT_DIR) / "dissociation_summary.csv").exists():
    import shutil
    shutil.copy2(Path(OUTPUT_DIR) / "dissociation_summary.csv", disso_top)
    shutil.copy2(Path(OUTPUT_DIR) / "extended_table1.csv", WORK / "extended_table1.csv")
    out("    (copied dissociation_summary.csv + extended_table1.csv to /kaggle/working/ top level)")

# ---- versions --------------------------------------------------------------
out("\n[versions]")
import importlib
for name in ("transformers", "accelerate", "bitsandbytes", "datasets", "sklearn",
             "matplotlib", "torch"):
    try:
        out(f"    {name}: {importlib.import_module(name).__version__}")
    except Exception:
        out(f"    {name}: (n/a)")

(WORK / "run_report.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"\n[report] wrote {WORK / 'run_report.txt'}")
print("\nNEXT:")
print("  * Resume: if any 'paused/deferred/skipped' tasks remain, Save Version, start a")
print("    fresh session, attach THIS run's saved output as a Dataset, and Run All — the")
print("    repo re-clones automatically and every completed unit is skipped.")
print("  * Persist results: 'Save Version' preserves /kaggle/working (extended tables,")
print("    scale_curves.png, raw judgments, checkpoints).")
print("  * If the clone failed: confirm the repo is pushed & public at REPO_GIT_URL with")
print("    Internet ON (or add a GITHUB_TOKEN secret / attach the repo as a Dataset), and")
print("    that the push included the src/utils.py MIRROR_MAX_MEMORY change (27B needs it).")
